# Creditability Prediction — Google Colab
Upload `german_credit.csv` when prompted. Run the single code cell to train three models, select the highest ROC-AUC model, save it with joblib and launch Gradio.

In [ ]:
!pip -q install gradio pandas scikit-learn joblib
from google.colab import files
import pathlib
print("Upload german_credit.csv")
uploaded = files.upload()
pathlib.Path('german_credit.csv').write_bytes(next(iter(uploaded.values())))
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

BASE = Path.cwd()
df = pd.read_csv(BASE / 'german_credit.csv')
X = df[['Age', 'Credit Amount']]
y = df['Creditability']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, min_samples_leaf=15, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=5, min_samples_leaf=10, random_state=42),
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
    results.append({'Model': name, 'Accuracy': accuracy_score(y_test, preds), 'F1': f1_score(y_test, preds), 'ROC-AUC': roc_auc_score(y_test, probs)})
results = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print(results.to_string(index=False))
winning_name = results.iloc[0]['Model']
joblib.dump(models[winning_name], BASE / 'creditability_model.joblib')
print(f'\nSaved: {winning_name} -> creditability_model.joblib')
results.to_csv(BASE / 'model_comparison.csv', index=False)

# Launch Gradio within Colab
import os
from pathlib import Path
import joblib
import pandas as pd
import gradio as gr

model = joblib.load('creditability_model.joblib')

def predict(age, credit_amount):
    inputs = pd.DataFrame([[age, credit_amount]], columns=['Age', 'Credit Amount'])
    predicted = int(model.predict(inputs)[0])
    probability = float(model.predict_proba(inputs)[0][list(model.classes_).index(1)])
    label = 'Good Creditability' if predicted == 1 else 'Bad Creditability'
    return label, f'{probability:.1%}'

with gr.Blocks(title='Creditability Prediction') as demo:
    gr.Markdown('# Creditability Prediction')
    gr.Markdown('Educational demonstration only — not for actual credit decisions.')
    age = gr.Number(label='Age', value=35, minimum=18, maximum=100, precision=0)
    amount = gr.Number(label='Credit Amount', value=3000, minimum=0)
    button = gr.Button('Predict Creditability', variant='primary')
    result = gr.Textbox(label='Prediction')
    confidence = gr.Textbox(label='Estimated Probability of Good Creditability')
    button.click(predict, inputs=[age, amount], outputs=[result, confidence])

if __name__ == '__main__':
    demo.launch(share=True)

# Download the trained model after testing:
# files.download('creditability_model.joblib')
